<a href="https://colab.research.google.com/github/buse-toklu/CENG467_Midterm/blob/main/q4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# CELL 1: Install required packages
# ============================================================
# Run once in your environment:
!pip uninstall torchtext torch -y -q
!pip install torch torchtext sacrebleu spacy nltk bert-score transformers sentencepiece -q
!python -m spacy download en_core_web_sm -q
!python -m spacy download de_core_news_sm -q


In [ ]:
# ============================================================
# CELL 2: Imports & reproducibility
# ============================================================
import warnings, random, math, time, textwrap
warnings.filterwarnings("ignore")

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torch.nn.utils.rnn import pad_sequence

from torchtext.datasets import Multi30k
from torchtext.data.utils import get_tokenizer
from torchtext.vocab import build_vocab_from_iterator

import sacrebleu
import nltk
from nltk.translate.meteor_score import meteor_score
from nltk.tokenize import word_tokenize
from bert_score import score as bert_score_fn

nltk.download("punkt",     quiet=True)
nltk.download("punkt_tab", quiet=True)
nltk.download("wordnet",   quiet=True)
nltk.download("omw-1.4",   quiet=True)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")


In [ ]:
# ============================================================
# CELL 3: Tokenizers (spaCy)
# ============================================================
token_de = get_tokenizer("spacy", language="de_core_news_sm")
token_en = get_tokenizer("spacy", language="en_core_web_sm")

def yield_tokens(data_iter, tokenizer, index):
    for sample in data_iter:
        yield tokenizer(sample[index])

In [ ]:
# ============================================================
# CELL 4: Vocabulary
# ============================================================
UNK, PAD, BOS, EOS = "<unk>", "<pad>", "<bos>", "<eos>"
SPECIALS = [UNK, PAD, BOS, EOS]

print("Building vocabularies …")
train_iter = Multi30k(split="train", language_pair=("de", "en"))
vocab_de   = build_vocab_from_iterator(
    yield_tokens(train_iter, token_de, 0), min_freq=2, specials=SPECIALS)
vocab_de.set_default_index(vocab_de[UNK])

train_iter = Multi30k(split="train", language_pair=("de", "en"))
vocab_en   = build_vocab_from_iterator(
    yield_tokens(train_iter, token_en, 1), min_freq=2, specials=SPECIALS)
vocab_en.set_default_index(vocab_en[UNK])

PAD_IDX = vocab_en[PAD]
BOS_IDX = vocab_en[BOS]
EOS_IDX = vocab_en[EOS]
SRC_PAD = vocab_de[PAD]
print(f"DE vocab: {len(vocab_de):,}  |  EN vocab: {len(vocab_en):,}")


In [ ]:
# ============================================================
# CELL 5: Preprocessing helpers
# ============================================================
def text_to_tensor(text, vocab, tokenizer):
    """Lowercase → tokenise → numericalize → prepend BOS, append EOS."""
    tokens = tokenizer(text.lower())
    ids    = [vocab[BOS]] + vocab(tokens) + [vocab[EOS]]
    return torch.tensor(ids, dtype=torch.long)

def collate_fn(batch):
    """
    Preprocessing steps applied per batch:
      1. Lowercase + spaCy tokenise each sentence
      2. Map tokens to integer indices (OOV → <unk>)
      3. Wrap with <bos> and <eos>
      4. Pad all sequences in the batch to the same length
    Output shapes: (src_len, B) and (tgt_len, B)  [time-first].
    """
    src_list, tgt_list = [], []
    for src_t, tgt_t in batch:
        src_list.append(text_to_tensor(src_t, vocab_de, token_de))
        tgt_list.append(text_to_tensor(tgt_t, vocab_en, token_en))
    src_b = pad_sequence(src_list, padding_value=SRC_PAD)
    tgt_b = pad_sequence(tgt_list, padding_value=PAD_IDX)
    return src_b, tgt_b

In [ ]:
# ============================================================
# CELL 6: DataLoaders
# ============================================================
BATCH_SIZE = 128

train_data = list(Multi30k(split="train", language_pair=("de", "en")))
val_data   = list(Multi30k(split="valid", language_pair=("de", "en")))
test_data  = list(Multi30k(split="test",  language_pair=("de", "en")))

train_loader = DataLoader(train_data, batch_size=BATCH_SIZE,
                          shuffle=True,  collate_fn=collate_fn)
val_loader   = DataLoader(val_data,   batch_size=BATCH_SIZE,
                          shuffle=False, collate_fn=collate_fn)
test_loader  = DataLoader(test_data,  batch_size=BATCH_SIZE,
                          shuffle=False, collate_fn=collate_fn)

print(f"Train: {len(train_loader)} batches | "
      f"Val: {len(val_loader)} | Test: {len(test_loader)}")

In [ ]:
# ============================================================
# CELL 7: MODEL 1 — Encoder (Bidirectional GRU)
# ============================================================
class Encoder(nn.Module):
    """
    Bidirectional GRU encoder.
    Processes source tokens in both directions so each hidden
    state captures left and right context.
    Returns:
        outputs : (src_len, B, 2*enc_hid)  — all hidden states
        hidden  : (B, dec_hid)             — initial decoder state
    """
    def __init__(self, input_dim, emb_dim, enc_hid, dec_hid, dropout):
        super().__init__()
        self.embedding = nn.Embedding(input_dim, emb_dim, padding_idx=SRC_PAD)
        self.rnn       = nn.GRU(emb_dim, enc_hid, bidirectional=True,
                                batch_first=False)
        self.fc        = nn.Linear(enc_hid * 2, dec_hid)
        self.dropout   = nn.Dropout(dropout)

    def forward(self, src):
        embedded        = self.dropout(self.embedding(src))     # (T, B, emb)
        outputs, hidden = self.rnn(embedded)                    # (T,B,2H),(2,B,H)
        hidden = torch.tanh(self.fc(
            torch.cat((hidden[-2], hidden[-1]), dim=1)))        # (B, dec_hid)
        return outputs, hidden

In [ ]:
# ============================================================
# CELL 8: MODEL 1 — Bahdanau Attention
# ============================================================
class BahdanauAttention(nn.Module):
    """
    Additive (Bahdanau) attention mechanism.
        e_ij  = v^T · tanh( W_s · s_{i-1}  +  W_h · h_j )
        α_ij  = softmax(e_ij)
        c_i   = Σ_j  α_ij · h_j

    α gives an alignment distribution over source positions,
    making the decoder focus on the most relevant source tokens
    at each generation step.
    """
    def __init__(self, enc_hid, dec_hid):
        super().__init__()
        self.attn = nn.Linear(enc_hid * 2 + dec_hid, dec_hid)
        self.v    = nn.Linear(dec_hid, 1, bias=False)

    def forward(self, hidden, enc_out):
        T = enc_out.shape[0]
        h = hidden.unsqueeze(1).repeat(1, T, 1)                # (B, T, dec_hid)
        e = enc_out.permute(1, 0, 2)                           # (B, T, 2*enc_hid)
        energy    = torch.tanh(self.attn(torch.cat((h, e), 2)))
        attention = self.v(energy).squeeze(2)                  # (B, T)
        return torch.softmax(attention, dim=1)


In [ ]:
# ============================================================
# CELL 9: MODEL 1 — RNN Decoder (GRU + Bahdanau)
# ============================================================
class RNNDecoder(nn.Module):
    def __init__(self, output_dim, emb_dim, enc_hid, dec_hid, dropout):
        super().__init__()
        self.embedding = nn.Embedding(output_dim, emb_dim, padding_idx=PAD_IDX)
        self.attention = BahdanauAttention(enc_hid, dec_hid)
        self.rnn       = nn.GRU(emb_dim + enc_hid * 2, dec_hid, batch_first=False)
        self.fc_out    = nn.Linear(dec_hid + enc_hid * 2 + emb_dim, output_dim)
        self.dropout   = nn.Dropout(dropout)

    def forward(self, token, hidden, enc_out):
        token = token.unsqueeze(0)                              # (1, B)
        emb   = self.dropout(self.embedding(token))            # (1, B, emb)
        a     = self.attention(hidden, enc_out).unsqueeze(1)   # (B, 1, T)
        ctx   = torch.bmm(a, enc_out.permute(1, 0, 2)).permute(1, 0, 2)  # (1,B,2H)
        out, hid = self.rnn(torch.cat((emb, ctx), dim=2), hidden.unsqueeze(0))
        pred  = self.fc_out(torch.cat(
            (out.squeeze(0), ctx.squeeze(0), emb.squeeze(0)), 1))
        return pred, hid.squeeze(0)


In [ ]:
# ============================================================
# CELL 10: MODEL 1 — Seq2Seq wrapper
# ============================================================
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, src, tgt, teacher_forcing_ratio=0.5):
        tgt_len, B  = tgt.shape
        vocab_size  = self.decoder.fc_out.out_features
        outputs     = torch.zeros(tgt_len, B, vocab_size).to(device)
        enc_out, hid = self.encoder(src)
        inp = tgt[0]
        for t in range(1, tgt_len):
            pred, hid  = self.decoder(inp, hid, enc_out)
            outputs[t] = pred
            inp = tgt[t] if random.random() < teacher_forcing_ratio \
                         else pred.argmax(1)
        return outputs

In [ ]:
# ============================================================
# CELL 11: MODEL 2 — Positional Encoding
# ============================================================
class PositionalEncoding(nn.Module):
    """
    Sinusoidal positional encoding (Vaswani et al., 2017).
        PE(pos, 2i)   = sin( pos / 10000^(2i / d_model) )
        PE(pos, 2i+1) = cos( pos / 10000^(2i / d_model) )
    Added to token embeddings so the Transformer can exploit word order
    without any recurrent or convolutional structure.
    """
    def __init__(self, d_model, max_len=512, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe  = torch.zeros(max_len, d_model)
        pos = torch.arange(max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float()
                        * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(1))            # (max_len, 1, d)

    def forward(self, x):
        return self.dropout(x + self.pe[:x.size(0)])

In [ ]:
# ============================================================
# CELL 12: MODEL 2 — Transformer (from scratch)
# ============================================================
class TransformerMT(nn.Module):
    """
    Encoder–Decoder Transformer built from scratch.

    Architecture differences vs Seq2Seq:
    • Multi-head self-attention (nhead=4, 4 layers) instead of GRU
    • All positions processed in parallel — O(1) distance between any two tokens
    • Sinusoidal positional encoding replaces recurrent order tracking
    • Causal (upper-triangular) mask prevents the decoder from attending
      to future target positions during training
    • No teacher-forcing quirks — standard cross-entropy on shifted targets
    """
    def __init__(self, src_vsz, trg_vsz,
                 d_model=256, nhead=4, num_enc=4, num_dec=4,
                 ffn_dim=512, dropout=0.1):
        super().__init__()
        self.src_emb = nn.Embedding(src_vsz, d_model, padding_idx=SRC_PAD)
        self.trg_emb = nn.Embedding(trg_vsz, d_model, padding_idx=PAD_IDX)
        self.src_pe  = PositionalEncoding(d_model, dropout=dropout)
        self.trg_pe  = PositionalEncoding(d_model, dropout=dropout)
        self.scale   = math.sqrt(d_model)

        enc_layer = nn.TransformerEncoderLayer(
            d_model, nhead, ffn_dim, dropout, batch_first=False, norm_first=True)
        dec_layer = nn.TransformerDecoderLayer(
            d_model, nhead, ffn_dim, dropout, batch_first=False, norm_first=True)
        self.encoder  = nn.TransformerEncoder(enc_layer, num_enc)
        self.decoder  = nn.TransformerDecoder(dec_layer, num_dec)
        self.fc_out   = nn.Linear(d_model, trg_vsz)
        self._init_weights()
    def _init_weights(self):
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

    def _src_pad_mask(self, src):
        return (src == SRC_PAD).T                              # (B, T)

    def _causal_mask(self, sz):
        return torch.triu(
            torch.ones(sz, sz, device=device), diagonal=1).bool()

    def encode(self, src):
        pad = self._src_pad_mask(src)
        x   = self.src_pe(self.src_emb(src) * self.scale)
        return self.encoder(x, src_key_padding_mask=pad), pad

    def decode(self, trg, memory, mem_pad):
        sz  = trg.size(0)
        cau = self._causal_mask(sz)
        x   = self.trg_pe(self.trg_emb(trg) * self.scale)
        out = self.decoder(x, memory,
                           tgt_mask=cau,
                           memory_key_padding_mask=mem_pad)
        return self.fc_out(out)

    def forward(self, src, trg):
        memory, mem_pad = self.encode(src)
        return self.decode(trg[:-1], memory, mem_pad)          # (T-1, B, V)


In [ ]:
# ============================================================
# CELL 13: Instantiate both models
# ============================================================
def count_params(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)

# Model 1
enc      = Encoder(len(vocab_de), 256, 512, 512, 0.5)
rnn_dec  = RNNDecoder(len(vocab_en), 256, 512, 512, 0.5)
seq2seq  = Seq2Seq(enc, rnn_dec).to(device)

def init_weights(m):
    for name, p in m.named_parameters():
        nn.init.normal_(p.data, 0, 0.01) if "weight" in name \
            else nn.init.constant_(p.data, 0)

seq2seq.apply(init_weights)
print(f"Seq2Seq (Bahdanau)    params: {count_params(seq2seq):>10,}")

# Model 2
tfm = TransformerMT(
    src_vsz=len(vocab_de), trg_vsz=len(vocab_en),
    d_model=256, nhead=4, num_enc=4, num_dec=4,
    ffn_dim=512, dropout=0.1
).to(device)
print(f"Transformer (scratch) params: {count_params(tfm):>10,}")

In [ ]:
# ============================================================
# CELL 14: Shared loss
# ============================================================
criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)

In [ ]:
# ============================================================
# CELL 15: Train Seq2Seq
# ============================================================
def _train_s2s(model, loader, opt):
    model.train(); total = 0
    for src, tgt in loader:
        src, tgt = src.to(device), tgt.to(device)
        opt.zero_grad()
        out  = model(src, tgt)
        V    = out.shape[-1]
        loss = criterion(out[1:].reshape(-1, V), tgt[1:].reshape(-1))
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step(); total += loss.item()
    return total / len(loader)

def _eval_s2s(model, loader):
    model.eval(); total = 0
    with torch.no_grad():
        for src, tgt in loader:
            src, tgt = src.to(device), tgt.to(device)
            out  = model(src, tgt, teacher_forcing_ratio=0.0)
            V    = out.shape[-1]
            total += criterion(out[1:].reshape(-1, V), tgt[1:].reshape(-1)).item()
    return total / len(loader)

N_EPOCHS = 10
opt_s2s  = optim.Adam(seq2seq.parameters())
best_s2s = float("inf")
print(f"\n--- Training Seq2Seq + Bahdanau Attention ({N_EPOCHS} epochs) ---")
print(f"{'Ep':>3} | {'Train':>8} | {'PPL':>8} | {'Val':>8} | {'PPL':>8} | {'s':>5}")
print("-" * 50)
for ep in range(1, N_EPOCHS + 1):
    t0 = time.time()
    tr = _train_s2s(seq2seq, train_loader, opt_s2s)
    vl = _eval_s2s(seq2seq, val_loader)
    if vl < best_s2s:
        best_s2s = vl
        torch.save(seq2seq.state_dict(), "seq2seq_best.pt")
    print(f"{ep:>3} | {tr:>8.4f} | {math.exp(tr):>8.2f} | "
          f"{vl:>8.4f} | {math.exp(vl):>8.2f} | {time.time()-t0:>4.1f}s")

In [ ]:
# ============================================================
# CELL 16: Train Transformer (from scratch)
# ============================================================
def _train_tfm(model, loader, opt):
    model.train(); total = 0
    for src, tgt in loader:
        src, tgt = src.to(device), tgt.to(device)
        opt.zero_grad()
        out  = model(src, tgt)                                 # (T-1, B, V)
        V    = out.shape[-1]
        loss = criterion(out.reshape(-1, V), tgt[1:].reshape(-1))
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step(); total += loss.item()
    return total / len(loader)

def _eval_tfm(model, loader):
    model.eval(); total = 0
    with torch.no_grad():
        for src, tgt in loader:
            src, tgt = src.to(device), tgt.to(device)
            out  = model(src, tgt)
            V    = out.shape[-1]
            total += criterion(out.reshape(-1, V), tgt[1:].reshape(-1)).item()
    return total / len(loader)

opt_tfm   = optim.Adam(tfm.parameters(), lr=5e-4, betas=(0.9, 0.98), eps=1e-9)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(opt_tfm, patience=3, factor=0.5)
best_tfm  = float("inf")
print(f"\n--- Training Transformer from Scratch ({N_EPOCHS} epochs) ---")
print(f"{'Ep':>3} | {'Train':>8} | {'PPL':>8} | {'Val':>8} | {'PPL':>8} | {'s':>5}")
print("-" * 50)
for ep in range(1, N_EPOCHS + 1):
    t0 = time.time()
    tr = _train_tfm(tfm, train_loader, opt_tfm)
    vl = _eval_tfm(tfm, val_loader)
    scheduler.step(vl)
    if vl < best_tfm:
        best_tfm = vl
        torch.save(tfm.state_dict(), "tfm_best.pt")
    print(f"{ep:>3} | {tr:>8.4f} | {math.exp(tr):>8.2f} | "
          f"{vl:>8.4f} | {math.exp(vl):>8.2f} | {time.time()-t0:>4.1f}s")


In [ ]:
# ============================================================
# CELL 17: Greedy inference
# ============================================================
def translate_seq2seq(sentence, model, max_len=50):
    """Greedy decoding — GRU Seq2Seq + Bahdanau Attention."""
    model.eval()
    tokens = token_de(sentence.lower())
    ids    = [vocab_de[BOS]] + vocab_de(tokens) + [vocab_de[EOS]]
    src    = torch.tensor(ids, dtype=torch.long).unsqueeze(1).to(device)
    with torch.no_grad():
        enc_out, hid = model.encoder(src)
    result, tok = [], torch.tensor([BOS_IDX]).to(device)
    for _ in range(max_len):
        with torch.no_grad():
            pred, hid = model.decoder(tok, hid, enc_out)
        nxt = pred.argmax(1).item()
        if nxt == EOS_IDX:
            break
        result.append(vocab_en.lookup_token(nxt))
        tok = torch.tensor([nxt]).to(device)
    return " ".join(result)

    def translate_transformer(sentence, model, max_len=50):
    """Greedy decoding — Transformer from scratch."""
    model.eval()
    tokens = token_de(sentence.lower())
    ids    = [vocab_de[BOS]] + vocab_de(tokens) + [vocab_de[EOS]]
    src    = torch.tensor(ids, dtype=torch.long).unsqueeze(1).to(device)
    with torch.no_grad():
        memory, mem_pad = model.encode(src)
    trg    = torch.tensor([[BOS_IDX]], dtype=torch.long).to(device)
    result = []
    for _ in range(max_len):
        with torch.no_grad():
            logits = model.decode(trg, memory, mem_pad)        # (T, 1, V)
        nxt = logits[-1, 0].argmax(-1).item()
        if nxt == EOS_IDX:
            break
        result.append(vocab_en.lookup_token(nxt))
        trg = torch.cat(
            [trg, torch.tensor([[nxt]], dtype=torch.long).to(device)], dim=0)
    return " ".join(result)

In [ ]:
# ============================================================
# CELL 18: Generate test-set translations
# ============================================================
seq2seq.load_state_dict(torch.load("seq2seq_best.pt", map_location=device))
tfm.load_state_dict(torch.load("tfm_best.pt",         map_location=device))

src_sentences = [p[0] for p in test_data]
ref_sentences = [p[1] for p in test_data]

print("\nGenerating translations …")
hyps_s2s = [translate_seq2seq(s, seq2seq) for s in src_sentences]
print(f"  Seq2Seq done          ({len(hyps_s2s)} sentences)")
hyps_tfm = [translate_transformer(s, tfm) for s in src_sentences]
print(f"  Transformer done      ({len(hyps_tfm)} sentences)")


In [ ]:
# ============================================================
# CELL 19: Multi-metric evaluation suite
# ============================================================
def compute_bleu(hypotheses, references):
    """
    Corpus-level BLEU (sacrebleu).
    Counts matching 1–4 grams with brevity penalty for short outputs.
    """
    return round(sacrebleu.corpus_bleu(hypotheses, [references]).score, 2)

def compute_chrf(hypotheses, references):
    """
    Corpus-level ChrF — character n-gram F-score (sacrebleu).
    More robust than BLEU for morphologically rich languages like German.
    """
    return round(sacrebleu.corpus_chrf(hypotheses, [references]).score, 2)

def compute_meteor(hypotheses, references):
    """
    Mean sentence-level METEOR (NLTK).
    Uses WordNet synonymy + Porter stemming to reward paraphrases.
    """
    scores = [
        meteor_score([word_tokenize(r.lower())], word_tokenize(h.lower()))
        for h, r in zip(hypotheses, references)
    ]
    return round(np.mean(scores) * 100, 2)
    def compute_bertscore(hypotheses, references):
    """
    BERTScore F1 via RoBERTa-large (real contextual embeddings).
    Measures semantic similarity — rewards meaning-equivalent paraphrases.
    """
    _, _, F1 = bert_score_fn(
        hypotheses, references,
        lang="en", model_type="roberta-large",
        verbose=False, device=str(device)
    )
    return round(F1.mean().item() * 100, 2)

print("\nComputing metrics … (BERTScore may take a moment)")

all_metrics = {}
for name, hyps in [
    ("Seq2Seq + Bahdanau Attn", hyps_s2s),
    ("Transformer (scratch)",   hyps_tfm),
]:
    all_metrics[name] = {
        "BLEU"      : compute_bleu(hyps, ref_sentences),
        "ChrF"      : compute_chrf(hyps, ref_sentences),
        "METEOR"    : compute_meteor(hyps, ref_sentences),
        "BERTScore" : compute_bertscore(hyps, ref_sentences),
    }

In [ ]:
sample_idx = 5
src_sample = src_sentences[sample_idx]
ref_sample = ref_sentences[sample_idx]
hyps_s2s_sample = hyps_s2s[sample_idx]
hyps_tfm_sample = hyps_tfm[sample_idx]

print(f"Source (DE): {src_sample}")
print(f"Reference (EN): {ref_sample}")
print(f"Seq2Seq Translation: {hyps_s2s_sample}")
print(f"Transformer Translation: {hyps_tfm_sample}")

# Discussion placeholder
print("\n--- Discussion ---")
print("Analyze differences in fluency, handling of rare words, or long-range dependencies here.")
print("For example, you might notice that one model produces more grammatically correct sentences or handles complex phrases better than the other.")